# 10-Agentic RL

对应 `trainer/train_agent.py`。在多轮 Tool-Use 上跑 GRPO / CISPO：模型先发 `<tool_call>`，环境（notebook 里用 mock）回 `<tool_response>`，再继续生成，最后用 `gt` 或规则打分。

主线数据：`agent_rl.jsonl` / `agent_rl_math.jsonl`。


In [ ]:
import os, sys, math, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
import torch
from torch import optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import PretrainDataset, SFTDataset, DPODataset, RLAIFDataset, AgentRLDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("../model")
print("device:", device, "vocab:", tokenizer.vocab_size)

import json, re


## 模板如何展开 tools


In [ ]:
ds = AgentRLDataset("./toydata/agent_data.jsonl", tokenizer)
item = ds[0]
print("gt =", item["gt"])
print(tokenizer.apply_chat_template(
    item["messages"], tokenize=False, add_generation_prompt=True, tools=item["tools"]
))


## Mock 工具执行

正式脚本里有天气/汇率/翻译/计算器等模拟器。这里只演示解析 `<tool_call>` 并回填。


In [ ]:
def parse_tool_calls(text):
    calls = []
    for m in re.findall(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL):
        try:
            calls.append(json.loads(m.strip()))
        except json.JSONDecodeError:
            pass
    return calls

def execute(name, args):
    if name == "calculate_math":
        expr = str(args.get("expression", "0")).replace("×", "*").replace("^", "**")
        return {"result": str(eval(expr, {"__builtins__": {}}, {}))}
    if name == "translate_text":
        table = {("你好世界", "english"): "Hello World"}
        return {"translated_text": table.get((args.get("text"), args.get("target_language")), args.get("text"))}
    return {"error": "unknown tool"}

demo = '<tool_call>{"name":"calculate_math","arguments":{"expression":"256 * 37"}}</tool_call>'
for call in parse_tool_calls(demo):
    result = execute(call["name"], call["arguments"])
    print("call", call)
    print("result", result)
    print("ok", result.get("result") == "9472")


## 多轮消息怎么接

assistant 写出 tool_call 后，追加 `role=tool` 的结果，再 `add_generation_prompt` 让模型说人话。奖励通常看最终答案是否命中 `gt`、是否合法调用、是否重复。


In [ ]:
messages = list(item["messages"])
messages.append({
    "role": "assistant",
    "content": "",
    "tool_calls": [{"name": "calculate_math", "arguments": {"expression": "256 * 37"}}],
})
messages.append({"role": "tool", "content": json.dumps({"result": "9472"}, ensure_ascii=False)})
print(tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, tools=item["tools"]
))


完整训练：`cd trainer && python train_agent.py`。推理可看 `scripts/eval_toolcall.py` 与 `scripts/web_demo.py` 的多轮 tool call。

和 SFT 的分工：SFT 教会格式，Agentic RL 用可校验奖励强化“该不该调用、参数对不对、最终答案对不对”。
